Build the stratified cat/dog image folders from the official CIFAR-10 splits, without replacing an existing export.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torchvision.datasets import CIFAR10

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "datasets.py").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not locate project src/datasets.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
DATA_ROOT = PROJECT_ROOT / "data" / "catdog"
CIFAR_ROOT = PROJECT_ROOT / "data"
CLASS_NAMES = {3: "cat", 5: "dog"}
EXPECTED_FOLDERS = [DATA_ROOT / split / name for split in ("train", "val", "test") for name in CLASS_NAMES.values()]

if DATA_ROOT.exists():
    missing = [str(path) for path in EXPECTED_FOLDERS if not path.is_dir()]
    if missing:
        raise FileExistsError(f"Existing export is incomplete; refusing to overwrite it. Missing: {missing}")
    print(f"Existing image export found; skipping export: {DATA_ROOT}")
else:
    cifar_train = CIFAR10(root=str(CIFAR_ROOT), train=True, download=True)
    cifar_test = CIFAR10(root=str(CIFAR_ROOT), train=False, download=True)
    selected = [index for index, label in enumerate(cifar_train.targets) if label in CLASS_NAMES]
    selected_labels = [cifar_train.targets[index] for index in selected]
    train_indices, val_indices = train_test_split(
        selected, test_size=0.2, random_state=SEED, stratify=selected_labels
    )

    for folder in EXPECTED_FOLDERS:
        folder.mkdir(parents=True, exist_ok=False)

    def export_images(dataset, indices, split):
        for index in indices:
            label = int(dataset.targets[index])
            class_name = CLASS_NAMES[label]
            destination = DATA_ROOT / split / class_name / f"{class_name}_{index:05d}.png"
            dataset.data[index].copy()
            from PIL import Image
            Image.fromarray(dataset.data[index]).save(destination, format="PNG")

    export_images(cifar_train, train_indices, "train")
    export_images(cifar_train, val_indices, "val")
    test_indices = [index for index, label in enumerate(cifar_test.targets) if label in CLASS_NAMES]
    export_images(cifar_test, test_indices, "test")
    print(f"Exported CIFAR-10 cat/dog images under {DATA_ROOT}")


71.6%
50.1%

Load every split with ImageFolder, create CSV manifests and custom datasets, verify identical samples, and compute training-channel statistics.

In [ ]:
import math

import pandas as pd
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

from src.datasets import CsvImageDataset

to_tensor = transforms.ToTensor()
image_datasets = {
    split: ImageFolder(DATA_ROOT / split, transform=to_tensor)
    for split in ("train", "val", "test")
}
class_to_idx = image_datasets["train"].class_to_idx
assert class_to_idx == {"cat": 0, "dog": 1}, class_to_idx
print("class_to_idx:", class_to_idx)

count_rows = []
csv_datasets = {}
for split, image_dataset in image_datasets.items():
    labels = [label for _, label in image_dataset.samples]
    counts = np.bincount(labels, minlength=2)
    count_rows.extend(
        {"split": split, "class": name, "images": int(counts[class_to_idx[name]])}
        for name in ("cat", "dog")
    )
    csv_path = DATA_ROOT / f"{split}.csv"
    manifest = pd.DataFrame(
        [
            {"path": Path(path).relative_to(DATA_ROOT).as_posix(), "label": label}
            for path, label in image_dataset.samples
        ],
        columns=["path", "label"],
    )
    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        if not existing.equals(manifest):
            raise FileExistsError(f"Existing manifest differs; refusing to overwrite: {csv_path}")
    else:
        manifest.to_csv(csv_path, index=False)
    csv_datasets[split] = CsvImageDataset(csv_path, transform=to_tensor)

counts_df = pd.DataFrame(count_rows)
display(counts_df)
batch_images, batch_labels = next(iter(DataLoader(image_datasets["train"], batch_size=64, shuffle=False, num_workers=0)))
print("ImageFolder batch shape:", tuple(batch_images.shape), "dtype:", batch_images.dtype)

for split in ("train", "val", "test"):
    image_dataset = image_datasets[split]
    csv_dataset = csv_datasets[split]
    assert image_dataset.class_to_idx == class_to_idx
    assert len(image_dataset) == len(csv_dataset)
    for index in range(min(20, len(image_dataset))):
        image_a, label_a = image_dataset[index]
        image_b, label_b = csv_dataset[index]
        assert label_a == label_b and torch.equal(image_a, image_b), (split, index)
print("First 20 ImageFolder and CSV dataset samples/labels match for every split.")

train_loader = DataLoader(image_datasets["train"], batch_size=256, shuffle=False, num_workers=0)
channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
pixel_count = 0
for images, _ in train_loader:
    channel_sum += images.sum(dim=(0, 2, 3))
    channel_squared_sum += (images ** 2).sum(dim=(0, 2, 3))
    pixel_count += images.size(0) * images.size(2) * images.size(3)
channel_mean = channel_sum / pixel_count
channel_std = torch.sqrt(channel_squared_sum / pixel_count - channel_mean ** 2)
print("Train per-channel mean:", channel_mean.tolist())
print("Train per-channel std:", channel_std.tolist())


Plot a labeled 2-by-6 grid of training images.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for index, axis in enumerate(axes.flat):
    image, label = image_datasets["train"][index]
    axis.imshow(image.permute(1, 2, 0).numpy())
    axis.set_title(image_datasets["train"].classes[label])
    axis.axis("off")
fig.tight_layout()
plt.show()
